# Topic 4: Pair RDDs — `reduceByKey`, `groupByKey`, `sortByKey`

> **Phase 2 → Week 2 → Topic 4**

---

## The Excel PivotTable Analogy

Imagine you have a spreadsheet of sales data:
```
Alice  | Electronics | 500
Bob    | Clothing    | 200
Alice  | Electronics | 350
Carol  | Electronics | 600
Bob    | Clothing    | 150
```

A PivotTable groups by a key column and aggregates values.
That's exactly what Pair RDDs do — distributed, across machines.

- **Key** = the column you group by (like the PivotTable row field)
- **Value** = the data you aggregate (like the PivotTable value field)
- **reduceByKey** = the aggregation function (sum, max, count, etc.)

---

## What is a Pair RDD?

A **Pair RDD** is an RDD where every element is a **2-tuple (key, value)**:

```python
# Regular RDD:
[1, 2, 3, 4, 5]

# Pair RDD:
[("Engineering", 95000), ("Marketing", 72000), ("Engineering", 88000)]
#  ────key─────  ─value─   ──────key──────────   ─────value──────
```

Pair RDDs unlock special operations:
- `reduceByKey()` — aggregate values per key
- `groupByKey()` — group all values per key (careful!)
- `sortByKey()` — sort by key
- `join()` — join two Pair RDDs on key
- `mapValues()` — apply function to values only
- `keys()` / `values()` — extract just keys or values
- `countByKey()` — count values per key

---

## Creating a Pair RDD

```python
# Method 1: Use map() to create (key, value) tuples
rdd = sc.parallelize(["apple", "banana", "avocado", "berry", "apricot"])
pair_rdd = rdd.map(lambda word: (word[0], word))  # first letter as key
# → [("a", "apple"), ("b", "banana"), ("a", "avocado"), ("b", "berry"), ("a", "apricot")]

# Method 2: Directly parallelize tuples
pair_rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4)])

# Method 3: SparkContext's specialized creators
sc.parallelize(zip(keys, values))
```

---

## `reduceByKey()` vs `groupByKey()` — The Most Important Distinction

This is asked in **every single Spark interview**. Know it cold.

### groupByKey — the naive (slow) way
```
Step 1: Shuffle ALL (key, value) pairs across network
Step 2: For each key, group all values into a list
Step 3: You then apply your aggregation function

Network traffic: ALL values for EVERY key cross the network
Memory risk: If one key has millions of values → executor OOM
```

### reduceByKey — the smart (fast) way
```
Step 1: Within each partition, partially combine values with the same key
        (called "map-side combine" or "pre-aggregation")
Step 2: Shuffle only the PARTIALLY reduced values (much less data!)
Step 3: Final reduce on the shuffled data

Network traffic: MUCH LESS — only partial results cross the network
```

**Visual with data = [('India',1), ('USA',1), ('India',1), ('India',1), ('USA',1)]:**

```
              groupByKey:                    reduceByKey:

Partition 0:  ('India',1),('USA',1)         ('India',1),('USA',1)
Partition 1:  ('India',1),('India',1),      ('India',1),('India',1),
              ('USA',1)                     ('USA',1)

  [Shuffle: send ALL values]     [Shuffle: send PARTIAL sums]
  India → [1, 1, 1] → 3 values  India → [1, 2] → 2 values (pre-reduced!)
  USA   → [1, 1]    → 2 values  USA   → [1, 1] → 2 values

  Total sent: 5 values           Total sent: 4 values
```

With millions of records the difference becomes enormous.

**Rule**: If you're going to aggregate after `groupByKey`, use `reduceByKey` instead.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week2 - Pair RDDs") \
    .master("local[4]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("SparkSession ready!")

## Part 1: Creating Pair RDDs

In [ ]:
# Method 1: map() to create (key, value) tuples
words = sc.parallelize(["apple", "banana", "avocado", "berry", "apricot", "blueberry"], 2)

# Group by first letter
by_letter = words.map(lambda w: (w[0], w))
print("By first letter:", by_letter.collect())

# Method 2: Parallelize tuples directly
sales = sc.parallelize([
    ("Engineering", 95000),
    ("Marketing",   72000),
    ("Engineering", 88000),
    ("Sales",       55000),
    ("Marketing",   68000),
    ("Engineering", 105000),
    ("Sales",       62000),
    ("Marketing",   78000),
], 3)
print("\nSales pair RDD (dept, salary):", sales.collect())

## Part 2: `reduceByKey()` — The Right Way to Aggregate

In [ ]:
# reduceByKey: apply a function to all values with the same key
# Function must be: (value, value) → value (same type in, same type out)

print("=== reduceByKey ===")

# Sum salaries per department
total_salary = sales.reduceByKey(lambda a, b: a + b)
print("Total salary per dept (sum):")
for dept, total in sorted(total_salary.collect()):
    print(f"  {dept:<15} ${total:,}")

# Max salary per department
max_salary = sales.reduceByKey(lambda a, b: max(a, b))
print("\nMax salary per dept:")
for dept, mx in sorted(max_salary.collect()):
    print(f"  {dept:<15} ${mx:,}")

# Count employees per department (map each to 1, then sum)
counts = sales.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)
print("\nEmployee count per dept:")
for dept, count in sorted(counts.collect()):
    print(f"  {dept:<15} {count}")

In [ ]:
# Word count — the classic Spark example
text = sc.parallelize([
    "the quick brown fox",
    "the fox jumped over",
    "the quick rabbit ran",
    "over the brown hill",
], 2)

word_counts = (
    text
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

print("Word count (sorted by frequency):")
for word, count in word_counts.collect():
    bar = '█' * count
    print(f"  {word:<10} {count}  {bar}")

## Part 3: `groupByKey()` — When You Actually Need the Lists

In [ ]:
# groupByKey: groups ALL values per key into an iterable (ResultIterable)
# Returns: [(key, <ResultIterable>), ...]
# You must wrap values in list() to see them

print("=== groupByKey ===")

by_dept = sales.groupByKey()
print("Grouped by dept:")
for dept, salaries_iter in sorted(by_dept.collect()):
    salaries = list(salaries_iter)  # must convert iterable to list
    print(f"  {dept:<15} {salaries}")

print()
print("When to use groupByKey:")
print("  ✓ When you need ALL values as a collection (e.g., to compute median)")
print("  ✓ When you need to do complex per-group processing")
print("  ✗ Never just to aggregate (sum/count/max) — use reduceByKey instead!")

# Example: computing the MEDIAN per department (needs all values)
# Note: you can't compute median incrementally, so groupByKey is needed here
import statistics
median_salary = by_dept.mapValues(lambda vals: statistics.median(list(vals)))
print("\nMedian salary per dept (valid use of groupByKey):")
for dept, median in sorted(median_salary.collect()):
    print(f"  {dept:<15} ${median:,.0f}")

In [ ]:
# PROOF: reduceByKey is faster than groupByKey + mapValues
import time

# Create large dataset
big_data = sc.parallelize(
    [("key_" + str(i % 10), 1) for i in range(200_000)], 4
)

t0 = time.time()
result1 = big_data.reduceByKey(lambda a, b: a + b).collect()
time_reduce = time.time() - t0

t0 = time.time()
result2 = big_data.groupByKey().mapValues(sum).collect()
time_group = time.time() - t0

print(f"Same 200k records, 10 unique keys:")
print(f"  reduceByKey: {time_reduce:.3f}s")
print(f"  groupByKey:  {time_group:.3f}s")
print(f"  Speedup: {time_group/time_reduce:.1f}x")
print(f"  Results match: {sorted(result1) == sorted(result2)}")

## Part 4: `sortByKey()` — Sort Pair RDD by Key

In [ ]:
# sortByKey sorts by the key (first element of tuple)
# This is a WIDE transformation (global sort requires shuffle)

monthly_revenue = sc.parallelize([
    ("2024-03", 450000),
    ("2024-01", 320000),
    ("2024-06", 510000),
    ("2024-02", 290000),
    ("2024-05", 480000),
    ("2024-04", 395000),
], 3)

print("Sorted by month (ascending):")
for month, rev in monthly_revenue.sortByKey(ascending=True).collect():
    bar = '█' * (rev // 50000)
    print(f"  {month}  ${rev:,}  {bar}")

print("\nSorted by month (descending):")
for month, rev in monthly_revenue.sortByKey(ascending=False).collect():
    print(f"  {month}  ${rev:,}")

print()
# Custom sort key — sort by the last 2 chars of month (the month number)
print("sortByKey with custom keyfunc (by month number only):")
sorted_by_month_num = monthly_revenue.sortByKey(keyfunc=lambda k: k.split("-")[1])
for month, rev in sorted_by_month_num.collect():
    print(f"  {month}  ${rev:,}")

## Part 5: `mapValues()` and `flatMapValues()` — Transform Values, Keep Keys

In [ ]:
# mapValues: apply function to values only, keys unchanged
# This is a NARROW transformation (no shuffle)

salaries_by_dept = sc.parallelize([
    ("Engineering", [95000, 88000, 105000]),
    ("Marketing",   [72000, 68000, 78000]),
    ("Sales",       [55000, 62000]),
])

print("=== mapValues ===")

# Sum of salaries per dept
print("Total salary per dept:")
for dept, total in salaries_by_dept.mapValues(sum).collect():
    print(f"  {dept:<15} ${total:,}")

# Average salary per dept
print("\nAverage salary per dept:")
for dept, avg in salaries_by_dept.mapValues(lambda v: sum(v)/len(v)).collect():
    print(f"  {dept:<15} ${avg:,.0f}")

# flatMapValues: expand each value into multiple key-value pairs
# One (key, [v1,v2,v3]) → [(key,v1), (key,v2), (key,v3)]
print("\nflatMapValues — expand list to individual (dept, salary) pairs:")
flat = salaries_by_dept.flatMapValues(lambda v: v)
for item in flat.collect():
    print(f"  {item}")

## Part 6: `join()`, `leftOuterJoin()`, `rightOuterJoin()`, `fullOuterJoin()`

In [ ]:
# Join two pair RDDs on their keys
# Result: (key, (value_from_rdd1, value_from_rdd2))

# Employee → (emp_id, name)
employees = sc.parallelize([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Carol"),
    (4, "Dave"),   # Dave has no salary record
])

# Salary → (emp_id, salary)
salaries_rdd = sc.parallelize([
    (1, 95000),
    (2, 88000),
    (3, 72000),
    (5, 60000),   # emp 5 has salary but no employee record
])

print("=== INNER JOIN (only matching keys in both) ===")
inner = employees.join(salaries_rdd)
for emp_id, (name, salary) in sorted(inner.collect()):
    print(f"  emp_id={emp_id}  {name:<8}  ${salary:,}")
print("  (Dave and emp_5 are EXCLUDED — no match in both)")

print("\n=== LEFT OUTER JOIN (all from left, None if no right match) ===")
left = employees.leftOuterJoin(salaries_rdd)
for emp_id, (name, salary) in sorted(left.collect()):
    print(f"  emp_id={emp_id}  {name:<8}  {f'${salary:,}' if salary else 'NO SALARY'}")
print("  (Dave included with None salary)")

print("\n=== RIGHT OUTER JOIN (all from right, None if no left match) ===")
right = employees.rightOuterJoin(salaries_rdd)
for emp_id, (name, salary) in sorted(right.collect()):
    print(f"  emp_id={emp_id}  {name or 'UNKNOWN':<8}  ${salary:,}")
print("  (emp_5 included with None name)")

print("\n=== FULL OUTER JOIN (all from both) ===")
full = employees.fullOuterJoin(salaries_rdd)
for emp_id, (name, salary) in sorted(full.collect()):
    print(f"  emp_id={emp_id}  {(name or 'UNKNOWN'):<8}  {f'${salary:,}' if salary else 'NO SALARY'}")

## Part 7: `keys()`, `values()`, `countByKey()`

In [ ]:
data = sc.parallelize([
    ("Engineering", 95000),
    ("Marketing",   72000),
    ("Engineering", 88000),
    ("Sales",       55000),
    ("Marketing",   68000),
])

print("keys():          ", sorted(data.keys().collect()))
print("keys().distinct():", sorted(data.keys().distinct().collect()))
print("values():        ", sorted(data.values().collect()))
print()

# countByKey: count number of values per key (returns Python dict)
# ⚠️ Returns to Driver — use only for low-cardinality keys
counts = data.countByKey()
print("countByKey():", dict(counts))

## Part 8: Real-World Pair RDD Pipeline

In [ ]:
# Real pipeline: process e-commerce transactions
# Input: raw transaction lines
# Output: top 3 categories by revenue, showing count and avg order value

transactions = sc.parallelize([
    "tx001,electronics,299.99,2024-01-15",
    "tx002,clothing,49.99,2024-01-15",
    "tx003,electronics,599.99,2024-01-16",
    "tx004,food,24.99,2024-01-16",
    "tx005,electronics,1299.99,2024-01-17",
    "tx006,clothing,89.99,2024-01-17",
    "tx007,food,15.99,2024-01-18",
    "tx008,electronics,449.99,2024-01-18",
    "tx009,clothing,79.99,2024-01-19",
    "tx010,food,34.99,2024-01-19",
    "tx011,electronics,799.99,2024-01-20",
    "tx012,clothing,129.99,2024-01-20",
], 3)

# Parse → create (category, amount) pairs → compute (total, count) per category
def parse_tx(line):
    parts = line.split(",")
    return (parts[1], float(parts[2]))

results = (
    transactions
    .map(parse_tx)                                              # (category, amount)
    .aggregateByKey(
        (0.0, 0),                                              # zero: (sum, count)
        lambda acc, x: (acc[0] + x, acc[1] + 1),              # seqOp within partition
        lambda a, b: (a[0] + b[0], a[1] + b[1])               # combOp between partitions
    )                                                           # → (category, (sum, count))
    .mapValues(lambda x: {                                      # compute stats
        "total": x[0],
        "count": x[1],
        "avg": x[0] / x[1]
    })
    .sortBy(lambda x: x[1]["total"], ascending=False)          # sort by revenue
)

print("E-commerce Revenue Summary:")
print(f"{'Category':<15} {'Revenue':>12} {'Orders':>8} {'Avg Order':>12}")
print("-" * 52)
for category, stats in results.collect():
    print(f"{category:<15} ${stats['total']:>10,.2f} {stats['count']:>8} ${stats['avg']:>10,.2f}")

## Interview Cheat Sheet

**Q: What is a Pair RDD?**
> A Pair RDD is an RDD where every element is a 2-tuple (key, value). It enables key-based operations like reduceByKey, groupByKey, join, sortByKey. You create them by mapping a regular RDD to tuples: `rdd.map(lambda x: (key_func(x), x))`.

**Q: When would you use `groupByKey` over `reduceByKey`?**
> Use `reduceByKey` whenever you're going to aggregate (sum, max, count). It pre-aggregates within each partition before shuffling, reducing network I/O significantly. Use `groupByKey` only when you genuinely need all values as a collection — for example, computing the median (which can't be done incrementally).

**Q: What does `mapValues()` do and why is it better than `map()` for pair RDDs?**
> `mapValues(f)` applies `f` to only the VALUE of each (key, value) pair, leaving the key unchanged. It's better than `map(lambda kv: (kv[0], f(kv[1])))` because Spark can tell the keys haven't changed, potentially avoiding hash recomputation and optimizing subsequent shuffles.

**Q: What is `aggregateByKey` and when would you use it?**
> `aggregateByKey` is like `aggregate` but per key. It takes a zero value, a per-partition accumulator function, and a cross-partition combiner. Use it when you need to compute an output of a different type than the value — e.g., given `(category, amount)` pairs, compute `(category, {sum, count, avg})` in one pass.

**Q: Explain how Spark join works on Pair RDDs.**
> Spark join on Pair RDDs shuffles both RDDs by their keys (hash partitioning) so all values with the same key end up on the same executor. Then it joins the co-located values. If one RDD is small, use `broadcast` (in DataFrame API) to avoid the shuffle of the small RDD.

## Exercises

1. From a list of `(country, city)` pairs, use `groupByKey` to group cities by country, then `mapValues` to sort the city list alphabetically.
2. Build a word frequency RDD from any paragraph. Use `sortByKey` to get words alphabetically, then `sortBy(lambda x: -x[1])` to get by frequency.
3. Create two Pair RDDs: `orders = [(order_id, customer_id)]` and `customers = [(customer_id, name)]`. Join them to get `(order_id, (customer_id, name))`.
4. Use `aggregateByKey` on the sales data to compute `{sum, count, max, min}` per department in a single pass.

In [ ]:
# Exercise 1: Group cities by country, sort city lists
city_data = sc.parallelize([
    ("India", "Mumbai"), ("USA", "New York"), ("India", "Delhi"),
    ("UK", "London"), ("India", "Bangalore"), ("USA", "Chicago"),
    ("UK", "Manchester"), ("USA", "Los Angeles"),
])

result = city_data.groupByKey().mapValues(lambda cities: sorted(list(cities)))
print("Exercise 1 — Cities by country:")
for country, cities in sorted(result.collect()):
    print(f"  {country}: {cities}")

In [ ]:
# Exercise 4: aggregateByKey for complete stats in one pass
import math

sales = sc.parallelize([
    ("Engineering", 95000), ("Marketing", 72000),
    ("Engineering", 88000), ("Sales", 55000),
    ("Marketing", 68000), ("Engineering", 105000),
    ("Sales", 62000), ("Marketing", 78000),
], 3)

zero = (0, 0, 0, math.inf, -math.inf)  # (sum, count, sq_sum, min, max)

def seq_op(acc, x):
    s, cnt, sq, mn, mx = acc
    return (s+x, cnt+1, sq+x*x, min(mn,x), max(mx,x))

def comb_op(a1, a2):
    return (a1[0]+a2[0], a1[1]+a2[1], a1[2]+a2[2], min(a1[3],a2[3]), max(a1[4],a2[4]))

stats = sales.aggregateByKey(zero, seq_op, comb_op) \
             .mapValues(lambda v: {
                 "count": v[1],
                 "sum": v[0],
                 "avg": v[0]/v[1],
                 "min": v[3],
                 "max": v[4]
             })

print("Exercise 4 — Department stats (one pass):")
for dept, s in sorted(stats.collect()):
    print(f"  {dept:<15} count={s['count']} avg=${s['avg']:,.0f} min=${s['min']:,} max=${s['max']:,}")